# 06 · Ablation

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

In [ ]:
import pickle
import random
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from neurosat import NeuroSAT, make_batch, to_device

## Settings

In [ ]:
SEED = 42
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
CLIP = 0.65
TRAIN_SUBSET = 5000
EVAL_ROUNDS = 128
LARGE_N = 60

CONFIGS = [
    {"dim": 32, "rounds": 26},
    {"dim": 64, "rounds": 26},
    {"dim": 128, "rounds": 26},
    {"dim": 128, "rounds": 8},
    {"dim": 128, "rounds": 16},
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data

In [ ]:
with open("train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("val.pkl", "rb") as f:
    val_data = pickle.load(f)

with open("test.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

random.seed(SEED)
train_subset = random.sample(train_data, TRAIN_SUBSET)
large_data = large_test_data[LARGE_N]

print("Train subset:", len(train_subset))
print("Val:         ", len(val_data))
print("Test:        ", len(test_data))
print(f"Test n={LARGE_N}:  ", len(large_data))

## Helpers

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()


def batches(data, shuffle):
    data = data.copy()
    if shuffle:
        random.shuffle(data)
    for i in range(0, len(data), BATCH_SIZE):
        yield to_device(make_batch(data[i:i + BATCH_SIZE]), device)


def accuracy(model, data, rounds):
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in batches(data, False):
            logits = model(batch, rounds)
            correct += ((logits > 0).float() == batch["labels"]).sum().item()
    return correct / len(data)

## Training

In [ ]:
def train(dim, rounds):
    random.seed(SEED)
    torch.manual_seed(SEED)

    model = NeuroSAT(dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = 0
    best_state = None
    val_history = []
    start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in batches(train_subset, True):
            logits = model(batch, rounds)
            loss = loss_fn(logits, batch["labels"])
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optimizer.step()

        val_acc = accuracy(model, val_data, rounds)
        val_history.append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"  epoch {epoch:2d} | val acc {val_acc:.3f}")

    model.load_state_dict(best_state)
    minutes = (time.perf_counter() - start) / 60
    return model, val_history, minutes

## Run all configurations

In [ ]:
results = {}

for config in CONFIGS:
    dim = config["dim"]
    rounds = config["rounds"]
    name = f"d{dim}_r{rounds}"
    print(f"Training {name}")

    model, val_history, minutes = train(dim, rounds)
    torch.save(model.state_dict(), f"ablation_{name}.pt")

    results[name] = {
        "dim": dim,
        "rounds": rounds,
        "params": sum(p.numel() for p in model.parameters()),
        "minutes": minutes,
        "val_history": val_history,
        "test_acc": accuracy(model, test_data, rounds),
        "test_acc_long": accuracy(model, test_data, EVAL_ROUNDS),
        "large_acc_long": accuracy(model, large_data, EVAL_ROUNDS),
    }

    print(f"  test acc {results[name]['test_acc']:.3f} | test acc @{EVAL_ROUNDS} {results[name]['test_acc_long']:.3f}")

## Results table

In [ ]:
print(f"{'config':>10s} | {'params':>9s} | {'minutes':>7s} | {'test acc':>8s} | {'test @' + str(EVAL_ROUNDS):>9s} | {'n=' + str(LARGE_N) + ' @' + str(EVAL_ROUNDS):>10s}")

for name, r in results.items():
    print(f"{name:>10s} | {r['params']:9,d} | {r['minutes']:7.1f} | {r['test_acc']:8.3f} | {r['test_acc_long']:9.3f} | {r['large_acc_long']:10.3f}")

## Plots

In [ ]:
dim_runs = sorted([r for r in results.values() if r["rounds"] == 26], key=lambda r: r["dim"])
round_runs = sorted([r for r in results.values() if r["dim"] == 128], key=lambda r: r["rounds"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot([r["dim"] for r in dim_runs], [r["test_acc"] for r in dim_runs], marker="o", label="26 rounds")
axes[0].plot([r["dim"] for r in dim_runs], [r["test_acc_long"] for r in dim_runs], marker="o", label=f"{EVAL_ROUNDS} rounds")
axes[0].set_xlabel("Hidden size")
axes[0].set_ylabel("Test accuracy")
axes[0].set_xticks([r["dim"] for r in dim_runs])
axes[0].legend()

axes[1].plot([r["rounds"] for r in round_runs], [r["test_acc"] for r in round_runs], marker="o", label="Same rounds as training")
axes[1].plot([r["rounds"] for r in round_runs], [r["test_acc_long"] for r in round_runs], marker="o", label=f"{EVAL_ROUNDS} rounds")
axes[1].set_xlabel("Training rounds")
axes[1].set_ylabel("Test accuracy")
axes[1].set_xticks([r["rounds"] for r in round_runs])
axes[1].legend()

for name, r in results.items():
    axes[2].plot(range(1, EPOCHS + 1), r["val_history"], label=name)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Validation accuracy")
axes[2].legend()

plt.tight_layout()
plt.savefig("ablation.png", dpi=150)
plt.show()

## Save

In [ ]:
with open("ablation_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved ablation_results.pkl and ablation.png")